# Feed-forward neural network training with checkpointing
#### Payload type: Low-granularity GPU workload + CPU-GPU synchronization + checkpoint I/O

In [ ]:
# imports
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import time
import os

In [ ]:
# We first keep on working with a simple 2d dataset that is a sin + noise.
x_data = torch.rand((100,1), requires_grad=False)*2*torch.pi
y_data = torch.sin(x_data) + 0.2 * torch.randn_like(x_data)

In [ ]:
# We split our dataset into train, validation and test set.
idxs = torch.randperm(len(x_data))
train_idxs, val_idxs, test_idxs = idxs[:50], idxs[50:80], idxs[80:]
max_train_idx = int(0.5*len(x_data))
x_train, y_train = x_data[train_idxs], y_data[train_idxs]
x_val, y_val = x_data[val_idxs], y_data[val_idxs]
x_test, y_test = x_data[test_idxs], y_data[test_idxs]

In [ ]:
# We use a simple neural network with one hidden layer and the relu nonlinearity.
class FeedForwardNN(nn.Module):
    def __init__(self, hidden_dim=100) -> None:
        super().__init__()
        self.input = nn.Linear(1, hidden_dim)
        self.hidden1 = nn.Linear(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim,1)

    def forward(self, x):
        x = self.input(x)
        x = F.relu(x)
        x = self.hidden1(x)
        x = F.relu(x)
        out = self.output(x)
        return out

In [ ]:
## Solution
# Hyperparameters.
lr = 2e-4
epochs = 20000

# Instantiate our model.
model = FeedForwardNN(1000)
# We use an optimizer so we do not have to take care of updating the parameters ourselves.
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
# PyTorch implements all kinds of loss functions you may want to use.
mse = torch.nn.MSELoss()
# We can print a model summary.
print(model)

# Bring model and data to GPU
model = model.to('cuda')
x_train, y_train, x_val, y_val = x_train.to('cuda'), y_train.to('cuda'), x_val.to('cuda'), y_val.to('cuda')

# reporting the loss over time
losses_train = []
losses_val = []
best_val_loss = np.inf

# create dir to save the model
save_path = '../model_checkpoints/e01/simple_nn/'
os.makedirs(save_path, exist_ok=True)

print('Training the neural network...')
t = time.time()
for epoch in range(epochs):
    # Compute the forward pass.
    y_predicted = model(x_train)
    # Compute the loss.
    loss = mse(y_train, y_predicted)
    # Reset and then compute gradients of the tree.
    optimizer.zero_grad()
    loss.backward()
    # Updating the parameters.
    optimizer.step()
    # Variables for plotting.
    losses_train.append(loss.detach().item())
    with torch.no_grad():
        losses_val.append(mse(model(x_val),y_val).item())
    # If we improve on our validation loss, we checkpoint the model as the new best model.
    if losses_val[-1] < best_val_loss:
        best_val_loss = losses_val[-1]
        torch.save(model,os.path.join(save_path,'best_model.pt'))
# after training is finished we also save our final model
torch.save(model,os.path.join(save_path,'final_model.pt'))
print(f'Training finished. Elapsed time: {time.time()-t:.02f} seconds.')

# plot losses and fit.
fig, [ax1,ax2] = plt.subplots(1,2, figsize=(10,5))
ax1.plot(losses_train, label="training loss")
ax1.plot(losses_val, label="validation loss")
ax1.set_ylabel('mse')
ax1.set_xlabel('epoch')
ax1.set_ylim([0,0.5])
ax1.legend()
ax2.scatter(x_train.detach().cpu().numpy(),y_train.detach().cpu().numpy(),c='b',s=5, label="train")
ax2.scatter(x_val.detach().cpu().numpy(),y_val.detach().cpu().numpy(),c='r',s=5, label='validation')
ax2.scatter(x_test,y_test,c='g',s=5, label='test')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
x_plot = torch.linspace(0.,2*torch.pi,200).unsqueeze(1)
y_plot_final = model(x_plot.to('cuda'))
best_model = torch.load(os.path.join(save_path,'best_model.pt'), weights_only=False)
y_plot_best = best_model(x_plot.to('cuda'))
ax2.plot(x_plot.detach().cpu().numpy(), y_plot_final.detach().cpu().numpy(),label='prediction (last model)')
ax2.plot(x_plot.detach().cpu().numpy(), y_plot_best.detach().cpu().numpy(),label='prediction (best model)')
ax2.legend()